# Experiment 9: Language Modeling and Evaluation: Statistical N-Gram Models vs Neural Language Models

### Problem Statement:
Design and evaluate language models using N-Gram techniques and Neural Language Models to predict the next word in a sequence and assess language generation capabilities.


In [1]:
import math
import random
import torch
import nltk
from nltk.lm import MLE, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.tokenize import word_tokenize
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import numpy as np

# Ensure seed consistency
random.seed(42)
torch.manual_seed(42)

print("Libraries for statistical and neural language modeling initialized.")


/Users/jeet/miniconda3/envs/ml_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries for statistical and neural language modeling initialized.


In [2]:
training_text = [
    "natural language processing allows computers to read and understand text.",
    "natural language processing models learn complex semantic patterns from data.",
    "deep learning neural networks achieve state of the art results.",
    "neural networks process unstructured data to make accurate predictions.",
    "computers utilize machine learning algorithms to solve optimization problems.",
    "artificial intelligence systems are transforming modern technology and society.",
    "natural language processing and machine learning enable conversational agents."
]

tokenized_corpus = [word_tokenize(sent.lower()) for sent in training_text]

# Prepare n-gram pipelines for n=2 (Bigram) and n=3 (Trigram)
n_bigram = 2
train_data_bi, vocab_bi = padded_everygram_pipeline(n_bigram, tokenized_corpus)
model_bigram = Laplace(n_bigram)
model_bigram.fit(train_data_bi, vocab_bi)

n_trigram = 3
train_data_tri, vocab_tri = padded_everygram_pipeline(n_trigram, tokenized_corpus)
model_trigram = Laplace(n_trigram)
model_trigram.fit(train_data_tri, vocab_tri)

print(f"Bigram Model Vocabulary Size  : {len(model_bigram.vocab)} symbols")
print(f"Trigram Model Vocabulary Size : {len(model_trigram.vocab)} symbols")

# Next-word conditional probability inspection
ctx = ("natural",)
prob_nlp = model_bigram.score("language", ctx)
prob_dl = model_bigram.score("learning", ctx)
print(f"\nConditional Probability P('language' | 'natural') = {prob_nlp:.4f}")
print(f"Conditional Probability P('learning' | 'natural') = {prob_dl:.4f}")


Bigram Model Vocabulary Size  : 53 symbols
Trigram Model Vocabulary Size : 53 symbols

Conditional Probability P('language' | 'natural') = 0.0714
Conditional Probability P('learning' | 'natural') = 0.0179


In [3]:
test_sentences = [
    "natural language processing models learn patterns.",
    "neural networks process unstructured data.",
    "quantum physics explains astronomical phenomena." # High perplexity expected (out of domain)
]

eval_records = []
for sent in test_sentences:
    tokens = word_tokenize(sent.lower())
    # Create test n-grams
    test_ngrams_bi = list(nltk.ngrams(tokens, n=2, pad_left=True, pad_right=True, left_pad_symbol='<s>', right_pad_symbol='</s>'))
    test_ngrams_tri = list(nltk.ngrams(tokens, n=3, pad_left=True, pad_right=True, left_pad_symbol='<s>', right_pad_symbol='</s>'))
    
    ppl_bi = model_bigram.perplexity(test_ngrams_bi)
    ppl_tri = model_trigram.perplexity(test_ngrams_tri)
    
    eval_records.append({
        "Test Sentence": sent,
        "Bigram PPL": round(ppl_bi, 2),
        "Trigram PPL": round(ppl_tri, 2)
    })

df_ppl = pd.DataFrame(eval_records)
print("--- Statistical Model Perplexity Evaluation ---")
print(df_ppl.to_string(index=False))

print("\n--- Statistical Text Generation ---")
print("Generated Bigram Sequence :", " ".join(model_bigram.generate(8, random_seed=42)))
print("Generated Trigram Sequence:", " ".join(model_trigram.generate(8, random_seed=42)))


--- Statistical Model Perplexity Evaluation ---
                                     Test Sentence  Bigram PPL  Trigram PPL
natural language processing models learn patterns.       21.67        23.88
        neural networks process unstructured data.       21.71        25.48
  quantum physics explains astronomical phenomena.       40.80        42.16

--- Statistical Text Generation ---
Generated Bigram Sequence : neural networks achieve state of the art results
Generated Trigram Sequence: natural language processing allows computers to read and


In [4]:
model_name = "distilbert/distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
neural_lm = AutoModelForCausalLM.from_pretrained(model_name)
neural_lm.eval()

def compute_transformer_perplexity(text, model, tokenizer):
    encodings = tokenizer(text, return_tensors='pt')
    input_ids = encodings.input_ids
    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss.item()
        perplexity = math.exp(loss)
    return loss, perplexity

neural_evals = []
for sent in test_sentences:
    loss, ppl = compute_transformer_perplexity(sent, neural_lm, tokenizer)
    neural_evals.append({
        "Sentence": sent,
        "Cross-Entropy Loss": round(loss, 3),
        "Neural Transformer PPL": round(ppl, 2)
    })

df_neural_eval = pd.DataFrame(neural_evals)
print("--- Neural Transformer (DistilGPT-2) Perplexity Benchmarks ---")
print(df_neural_eval.to_string(index=False))


Loading weights:   0%|                                  | 0/76 [00:00<?, ?it/s]

Loading weights: 100%|███████████████████████| 76/76 [00:00<00:00, 9002.69it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


--- Neural Transformer (DistilGPT-2) Perplexity Benchmarks ---
                                          Sentence  Cross-Entropy Loss  Neural Transformer PPL
natural language processing models learn patterns.               8.148                 3457.86
        neural networks process unstructured data.               5.636                  280.21
  quantum physics explains astronomical phenomena.               6.924                 1016.85


In [5]:
prompt = "Natural Language Processing algorithms"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

# 1. Greedy Search
out_greedy = neural_lm.generate(input_ids, max_length=28, do_sample=False)
gen_greedy = tokenizer.decode(out_greedy[0], skip_special_tokens=True)

# 2. Temperature Sampling (High Temperature = 1.3)
out_temp = neural_lm.generate(input_ids, max_length=28, do_sample=True, temperature=1.3)
gen_temp = tokenizer.decode(out_temp[0], skip_special_tokens=True)

# 3. Top-K Sampling (K = 40)
out_topk = neural_lm.generate(input_ids, max_length=28, do_sample=True, top_k=40, temperature=0.8)
gen_topk = tokenizer.decode(out_topk[0], skip_special_tokens=True)

# 4. Top-p (Nucleus) Sampling (p = 0.90)
out_topp = neural_lm.generate(input_ids, max_length=28, do_sample=True, top_p=0.90, temperature=0.8)
gen_topp = tokenizer.decode(out_topp[0], skip_special_tokens=True)

decoding_comparison = [
    {"Strategy": "1. Greedy Search", "Generated Output": gen_greedy},
    {"Strategy": "2. High Temp (T=1.3)", "Generated Output": gen_temp},
    {"Strategy": "3. Top-K (K=40)", "Generated Output": gen_topk},
    {"Strategy": "4. Top-p Nucleus (p=0.9)", "Generated Output": gen_topp}
]

df_decoding = pd.DataFrame(decoding_comparison)
print("--- Text Generation Decoding Strategies Comparison ---")
for idx, row in df_decoding.iterrows():
    print(f"[{row['Strategy']}]\n  '{row['Generated Output']}'\n")


--- Text Generation Decoding Strategies Comparison ---
[1. Greedy Search]
  'Natural Language Processing algorithms are used to create a language that is not only readable but also readable.








'

[2. High Temp (T=1.3)]
  'Natural Language Processing algorithms: What does a language expect in languages like JavaScript or Google Chrome?"



As my research in Clojure continued'

[3. Top-K (K=40)]
  'Natural Language Processing algorithms are the most widely used tool for identifying words in the corpus. They are applied to most languages and languages, such as'

[4. Top-p Nucleus (p=0.9)]
  'Natural Language Processing algorithms are designed to give us the most accurate and accurate results when interpreting our speech patterns.






'

